# MarketPulse — PySpark ETL

Cleans raw tables and engineers features for downstream SQL analytics, statistics, A/B testing, and churn modeling.

Cleaning rules are based on findings from `01_data_ingestion.ipynb` (see local repo).

## 1. Setup — Load Raw Tables from S3

Reconnects to S3 using the same boto3 pattern as the ingestion notebook, and loads all 8 raw tables into Spark DataFrames for processing.

In [0]:
import boto3
import pandas as pd
from io import StringIO
from pyspark.sql import functions as F

s3 = boto3.client(
    "s3",
    aws_access_key_id="ACCESS_KEY_ID",
    aws_secret_access_key="SECRET_ACCESS_KEY",
    region_name="ap-southeast-2"
)

BUCKET = "marketpulse-narjeena-2026"

files = {
    "customers": "raw/customers/olist_customers_dataset.csv",
    "orders": "raw/orders/olist_orders_dataset.csv",
    "order_items": "raw/order_items/olist_order_items_dataset.csv",
    "products": "raw/products/olist_products_dataset.csv",
    "payments": "raw/payments/olist_order_payments_dataset.csv",
    "reviews": "raw/reviews/olist_order_reviews_dataset.csv",
    "sessions": "raw/sessions/sessions.csv",
    "experiments": "raw/experiments/experiments.csv",
}

dfs = {}
for name, key in files.items():
    obj = s3.get_object(Bucket=BUCKET, Key=key)
    pdf = pd.read_csv(StringIO(obj["Body"].read().decode("utf-8")))
    dfs[name] = spark.createDataFrame(pdf)

customers = dfs["customers"]
orders = dfs["orders"]
order_items = dfs["order_items"]
products = dfs["products"]
payments = dfs["payments"]
reviews = dfs["reviews"]
sessions = dfs["sessions"]
experiments = dfs["experiments"]

print("All tables loaded.")

All tables loaded.


## 2. Clean Orders

Converts date columns from string to proper timestamp type, and adds an `is_delivered` boolean flag instead of filling null delivery dates — since a null delivery date is meaningful (order cancelled or still in transit), not an error to be patched over.

In [0]:
orders_clean = (
    orders
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp"))
    .withColumn("order_approved_at", F.to_timestamp("order_approved_at"))
    .withColumn("order_delivered_carrier_date", F.to_timestamp("order_delivered_carrier_date"))
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date"))
    .withColumn("is_delivered", F.col("order_status") == "delivered")
)

orders_clean.select(
    "order_id", "order_status", "is_delivered",
    "order_purchase_timestamp", "order_delivered_customer_date"
).show(5)

+--------------------+------------+------------+------------------------+-----------------------------+
|            order_id|order_status|is_delivered|order_purchase_timestamp|order_delivered_customer_date|
+--------------------+------------+------------+------------------------+-----------------------------+
|e481f51cbdc54678b...|   delivered|        true|     2017-10-02 10:56:33|          2017-10-10 21:25:13|
|53cdb2fc8bc7dce0b...|   delivered|        true|     2018-07-24 20:41:37|          2018-08-07 15:27:45|
|47770eb9100c2d0c4...|   delivered|        true|     2018-08-08 08:38:49|          2018-08-17 18:06:29|
|949d5b44dbf5de918...|   delivered|        true|     2017-11-18 19:28:06|          2017-12-02 00:28:42|
|ad21c59c0840e6cb8...|   delivered|        true|     2018-02-13 21:18:39|          2018-02-16 18:17:02|
+--------------------+------------+------------+------------------------+-----------------------------+
only showing top 5 rows


## 3. Clean Products

Fills missing `product_category_name` with `"unknown"` rather than dropping those rows, to avoid silently losing revenue in later aggregations. Drops the 2 rows missing physical dimensions, since that's a trivial and safe loss.

In [0]:
products_clean = (
    products
    .fillna({"product_category_name": "unknown"})
    .dropna(subset=["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"])
)

print(f"Original: {products.count()} rows, Cleaned: {products_clean.count()} rows")
products_clean.select("product_id", "product_category_name", "product_weight_g").show(5)

Original: 32951 rows, Cleaned: 32949 rows
+--------------------+---------------------+----------------+
|          product_id|product_category_name|product_weight_g|
+--------------------+---------------------+----------------+
|1e9e8ef04dbcff454...|           perfumaria|           225.0|
|3aa071139cb16b67c...|                artes|          1000.0|
|96bd76ec8810374ed...|        esporte_lazer|           154.0|
|cef67bcfe19066a93...|                bebes|           371.0|
|9dc1a7de274444849...| utilidades_domest...|           625.0|
+--------------------+---------------------+----------------+
only showing top 5 rows


## 4. Sanity Check — Verify Cleaning

Confirms the `is_delivered` flag and order status counts match what was found during initial exploration (see `01_data_ingestion.ipynb`).

In [0]:
orders_clean.groupBy("order_status", "is_delivered").count().orderBy(F.desc("count")).show()

+------------+------------+-----+
|order_status|is_delivered|count|
+------------+------------+-----+
|   delivered|        true|96478|
|     shipped|       false| 1107|
|    canceled|       false|  625|
| unavailable|       false|  609|
|    invoiced|       false|  314|
|  processing|       false|  301|
|     created|       false|    5|
|    approved|       false|    2|
+------------+------------+-----+



## 5. Build Unified Transactions Table

Joins orders, order_items, products, and payments into a single transaction-level table — this becomes the base for all downstream SQL analytics, RFM feature engineering, and the churn model.

Only `delivered` orders are included here, since cancelled/undelivered orders shouldn't count toward revenue or customer value metrics. Order volume and cancellation-rate analysis (using the full unfiltered `orders_clean` table) will be handled separately in the SQL analytics notebook.

In [0]:
transactions = (
    orders_clean.filter(F.col("is_delivered") == True)
    .join(order_items, on="order_id", how="inner")
    .join(products_clean, on="product_id", how="inner")
    .join(
        payments.groupBy("order_id").agg(
            F.sum("payment_value").alias("total_payment_value"),
            F.max("payment_installments").alias("max_installments")
        ),
        on="order_id", how="left"
    )
)

print(f"Transactions row count: {transactions.count()}")
transactions.select(
    "order_id", "customer_id", "product_category_name",
    "price", "freight_value", "total_payment_value",
    "order_purchase_timestamp"
).show(5)

Transactions row count: 110179
+--------------------+--------------------+---------------------+-----+-------------+-------------------+------------------------+
|            order_id|         customer_id|product_category_name|price|freight_value|total_payment_value|order_purchase_timestamp|
+--------------------+--------------------+---------------------+-----+-------------+-------------------+------------------------+
|00010242fe8c5a6d1...|3ce436f183e68e078...|           cool_stuff| 58.9|        13.29|              72.19|     2017-09-13 08:59:02|
|00018f77f2f0320c5...|f6dd3ec061db4e398...|             pet_shop|239.9|        19.93|             259.83|     2017-04-26 10:53:06|
|000229ec398224ef6...|6489ae5e4333f3693...|     moveis_decoracao|199.0|        17.87|             216.87|     2018-01-14 14:33:31|
|00024acbcdf0a6daa...|d4eb9395c8c0431ee...|           perfumaria|12.99|        12.79|              25.78|     2018-08-08 10:00:35|
|00042b26cf59d7ce6...|58dbd0b2d70206bf4...|   ferram

## 6. Customer-Level Feature Engineering (RFM)

Aggregates the transaction table to one row per customer, computing Recency, Frequency, and Monetary value — the standard feature set for churn and customer-value analysis. These features feed directly into the churn model (Phase 9) and customer segmentation in Power BI (Phase 10).

In [0]:
# Get one row per order (not per item) with order-level payment total,
# to avoid double-counting revenue for orders with multiple items
order_level = (
    transactions
    .select("order_id", "customer_id", "order_purchase_timestamp", "total_payment_value")
    .dropDuplicates(["order_id"])
)

max_date = order_level.agg(F.max("order_purchase_timestamp")).collect()[0][0]

customer_features = (
    order_level.groupBy("customer_id")
    .agg(
        F.max("order_purchase_timestamp").alias("last_purchase_date"),
        F.countDistinct("order_id").alias("frequency"),
        F.sum("total_payment_value").alias("monetary_value"),
        F.avg("total_payment_value").alias("avg_order_value")
    )
    .withColumn(
        "recency_days",
        F.datediff(F.lit(max_date), F.col("last_purchase_date"))
    )
)

print(f"Unique customers: {customer_features.count()}")
customer_features.orderBy(F.desc("monetary_value")).show(5)

Unique customers: 96462
+--------------------+-------------------+---------+--------------+---------------+------------+
|         customer_id| last_purchase_date|frequency|monetary_value|avg_order_value|recency_days|
+--------------------+-------------------+---------+--------------+---------------+------------+
|1617b1357756262bf...|2017-09-29 15:24:52|        1|      13664.08|       13664.08|         334|
|ec5b2ba62e5743423...|2018-07-15 14:49:44|        1|       7274.88|        7274.88|          45|
|c6e2731c5b391845f...|2017-02-12 20:37:36|        1|       6929.31|        6929.31|         563|
|f48d464a0baaea338...|2018-07-25 18:10:17|        1|       6922.21|        6922.21|          35|
|3fd6777bbce08a352...|2017-05-24 18:14:34|        1|       6726.66|        6726.66|         462|
+--------------------+-------------------+---------+--------------+---------------+------------+
only showing top 5 rows


## 7. Extend Features — Delivery Time & Category Diversity

Adds average delivery time per customer (purchase-to-delivery duration) and count of distinct product categories purchased — both useful behavioral signals for the churn model and later correlation analysis.

In [0]:
delivery_features = (
    orders_clean.filter(F.col("is_delivered") == True)
    .withColumn(
        "delivery_days",
        F.datediff("order_delivered_customer_date", "order_purchase_timestamp")
    )
    .groupBy("customer_id")
    .agg(F.avg("delivery_days").alias("avg_delivery_days"))
)

category_diversity = (
    transactions
    .groupBy("customer_id")
    .agg(F.countDistinct("product_category_name").alias("distinct_categories"))
)

customer_features = (
    customer_features
    .join(delivery_features, on="customer_id", how="left")
    .join(category_diversity, on="customer_id", how="left")
)

customer_features.show(5)

+--------------------+-------------------+---------+--------------+---------------+------------+-----------------+-------------------+
|         customer_id| last_purchase_date|frequency|monetary_value|avg_order_value|recency_days|avg_delivery_days|distinct_categories|
+--------------------+-------------------+---------+--------------+---------------+------------+-----------------+-------------------+
|c5a92db38654f2f74...|2018-05-08 19:38:58|        1|        406.26|         406.26|         113|              6.0|                  1|
|62a8b4a0d10a8fa0f...|2018-08-20 18:55:21|        1|        792.17|         792.17|           9|              7.0|                  1|
|1108051163387bf4b...|2017-10-29 13:46:57|        1|        146.46|         146.46|         304|             34.0|                  1|
|cd83239e30889569d...|2018-03-23 01:42:47|        1|        108.59|         108.59|         159|             18.0|                  1|
|bff79ac2dd5a37352...|2017-08-18 18:55:02|        1|   

## 8. Save Processed Tables

Persists the cleaned transaction-level and customer-level feature tables as Delta tables, so they can be reused directly in the SQL analytics, statistics, A/B testing, and churn modeling notebooks without re-running this ETL pipeline each time.

In [0]:
transactions.write.format("delta").mode("overwrite").saveAsTable("marketpulse_transactions")
customer_features.write.format("delta").mode("overwrite").saveAsTable("marketpulse_customer_features")
orders_clean.write.format("delta").mode("overwrite").saveAsTable("marketpulse_orders_clean")
sessions.write.format("delta").mode("overwrite").saveAsTable("marketpulse_sessions")
experiments.write.format("delta").mode("overwrite").saveAsTable("marketpulse_experiments")

print("All tables saved.")

All tables saved.


## Summary

This notebook took the raw Olist tables plus synthetic sessions/experiments data and produced five reusable Delta tables:

- **marketpulse_orders_clean** — orders with proper datetime types and an `is_delivered` flag
- **marketpulse_transactions** — order-item level table joined with products and aggregated payments, filtered to delivered orders only
- **marketpulse_customer_features** — one row per customer with RFM features (recency, frequency, monetary value), average order value, average delivery time, and category diversity
- **marketpulse_sessions** / **marketpulse_experiments** — synthetic tables saved as-is for use in funnel and A/B testing analysis

A join/aggregation bug was caught and fixed during this process: summing order-level payment totals directly from the item-level transactions table double-counted revenue for multi-item orders. Fixed by deduplicating to one row per order before aggregating to the customer level (see NOTES.md).

**Next step:** `03_sql_analysis` — build the analytical SQL layer (monthly revenue, cohort retention, conversion funnel, customer/product rankings) on top of these Delta tables.